# Research 02 — DPO loss 手写 + Debug

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/research/02-dpo-loss.ipynb)

- **日期**：2026-10-02
- **定位**：Research PyTorch 轮次的正主 —— 电话点名的实操形式就是 *real DPO debugging*：给你一段能跑但训不动的代码，现场定位。
- **环境**：纯 PyTorch，CPU 即可（toy LM，秒级跑完）。

配套笔记：[02-dpo-loss.md](02-dpo-loss.md)


## 今日目标

1. 手写 `get_batch_logps`（shift + mask + **求和**）和 `dpo_loss`，每一步都知道在干什么。
2. 用一个 toy LM 把 DPO 训出可见的 reward margin。
3. 亲手踩三个经典 bug：符号写反、mean 代替 sum、ref 没 detach —— 看症状、定位、修复。

先记住报警信号：**reward accuracy < 0.5、loss 冻在 0.693（ln2）、训完模型只会说短句** —— 分别对应这三个 bug。


## 1. DPO 公式（先对齐记号）

$ y_w $ chosen、$ y_l $ rejected、$ \beta $ 控制与 ref 的 KL 强度：

$$ \mathcal{L}_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right] $$

括号里是 implicit reward 的差：$ \hat{r}(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)} $ 。accuracy 看 margin 的符号：$ \hat{r}(y_w) > \hat{r}(y_l) $ 的比例。


## 2. 命门：`get_batch_logps`

自回归 log-prob：$ \log \pi(y|x) = \sum_t \log p(y_t | x, y_{<t}) $ 。代码上三个动作：

1. logits 与 labels **错开一位**（位置 $ t $ 的 logits 预测 token $ t+1 $ ）
2. `log_softmax` 后 **gather** 出目标 token 的 log-prob
3. 按 mask **求和**（不是平均 —— 平均会给长序列系统性打折）


In [ ]:
import torch
import torch.nn.functional as F

def get_batch_logps(logits, labels):
    """把模型输出换算成每条序列的 log 概率（DPO / RLHF 的地基函数）。

    入参:
      logits: (B, T, V) float —— 模型在每个位置对词表 V 的未归一化打分；
              约定位置 t 的输出是"下一个 token 是什么"的预测。
      labels: (B, T) long —— 真实 token id（prompt + completion 拼成的整条序列）；
              不算 loss 的位置（prompt、padding）填 -100。
    返回:
      (B,) float —— 每条序列的 log 概率：completion 部分各 token 的 log p 之和
              （prompt 位置被 mask 排除）。

    本 notebook 的实例：make_batch 造出 B=32 条、T=12（4 prompt + 8 response）、V=16。
    """
    logits = logits[:, :-1, :]          # 位置 t 预测 token t+1
    labels = labels[:, 1:]
    logp = F.log_softmax(logits, dim=-1)
    token_logp = torch.gather(logp, 2, labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    mask = (labels != -100).float()
    return (token_logp * mask).sum(-1)  # ← 求和，不是平均

# 快速自检：均匀分布下，每个 token 的 logp 应为 -log(V)
torch.manual_seed(0)
logits = torch.zeros(2, 5, 8)
labels = torch.tensor([[0, 1, 2, 3, -100],
                       [0, 4, 5, 6, -100]])
out = get_batch_logps(logits, labels)
print('均匀分布 logp 自检:', [round(v, 3) for v in out.tolist()], ' 期望:', [round(-3 * torch.log(torch.tensor(8.)).item(), 3)] * 2)


## 3. `dpo_loss` + toy LM 训练

模型用最小的自回归 LM（embedding → 线性头，bigram 依赖）就够验证 DPO 的全部机制。ref 是 policy 的**冻结副本**。


In [ ]:
import copy

class TinyLM(torch.nn.Module):
    """最小语言模型：Embedding + 线性头（没有 transformer 层，只为跑通 logp 管线）。
    __init__: vocab 词表大小（默认 16），d embedding 维度（默认 32）。
    forward: ids (B, T) long → logits (B, T, vocab) float。"""
    def __init__(self, vocab=16, d=32):
        super().__init__()
        self.emb = torch.nn.Embedding(vocab, d)
        self.head = torch.nn.Linear(d, vocab)
    def forward(self, ids):
        return self.head(self.emb(ids))

def dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1):
    """DPO loss（闭式偏好优化）。

    入参（全部是 (B,) float，来自 get_batch_logps）:
      pi_w / pi_l:   当前 policy 对 chosen / rejected 整条序列的 log 概率
      ref_w / ref_l: 冻结的 ref 模型对 chosen / rejected 的 log 概率
      beta: KL 强度（隐式 reward 的缩放系数），越大越不许 policy 远离 ref
    返回: (loss 标量, chosen_reward (B,), rejected_reward (B,))
      reward = beta * (pi − ref)，即"这条序列相对 ref 的对数概率涨幅"。
    """
    pi_logratios = pi_w - pi_l
    ref_logratios = ref_w - ref_l
    logits = beta * (pi_logratios - ref_logratios)
    loss = -F.logsigmoid(logits).mean()
    with torch.no_grad():
        chosen_r = beta * (pi_w - ref_w)
        rejected_r = beta * (pi_l - ref_l)
    return loss, chosen_r, rejected_r


In [ ]:
torch.manual_seed(0)

# 合成偏好数据：prompt 4 token，response 8 token
# chosen 的 response 多由 {5,6,7} 组成，rejected 多由 {9,10,11} 组成
def make_batch(B=32):
    """造一批玩具偏好对。

    B: batch 条数。返回 ((ids_w, lab_w), (ids_l, lab_l))，四个张量形状都是 (B, 12)：
      ids 是 token id（前 4 个 prompt + 后 8 个 response）；
      lab 是训练标签，prompt 位置填 -100、response 位置复制 ids。
    偏好信号藏在分布里：chosen 的 response 只从 {5, 6, 7} 号 token 采，
    rejected 只从 {9, 10, 11} 采 —— 模型要学的就是这个区别。
    """
    prompt = torch.randint(0, 4, (B, 4))
    chosen_resp = torch.multinomial(torch.tensor([0.]*5 + [0.3, 0.4, 0.3] + [0.]*8), B * 8, replacement=True).view(B, 8)
    rejected_resp = torch.multinomial(torch.tensor([0.]*9 + [0.3, 0.4, 0.3] + [0.]*5), B * 8, replacement=True).view(B, 8)
    def pack(resp):
        ids = torch.cat([prompt, resp], dim=1)
        labels = ids.clone()
        labels[:, :4] = -100          # prompt 不进 log-prob
        return ids, labels
    return pack(chosen_resp), pack(rejected_resp)

policy = TinyLM()
ref = copy.deepcopy(policy)
for p in ref.parameters():
    p.requires_grad_(False)          # ← ref 冻结：detach 的等价物（参数级）

opt = torch.optim.Adam(policy.parameters(), lr=5e-3)
for step in range(300):
    (ids_w, lab_w), (ids_l, lab_l) = make_batch()
    pi_w = get_batch_logps(policy(ids_w), lab_w)
    pi_l = get_batch_logps(policy(ids_l), lab_l)
    with torch.no_grad():
        ref_w = get_batch_logps(ref(ids_w), lab_w)
        ref_l = get_batch_logps(ref(ids_l), lab_l)
    loss, cr, rr = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 60 == 0:
        acc = (cr > rr).float().mean().item()
        print(f'step {step:3d}  loss {loss.item():.4f}  margin {(cr-rr).mean().item():+.3f}  acc {acc:.2f}')


训练健康时应该看到：loss 从 0.69 往下掉、margin 转正并变大、accuracy 爬到接近 1.0。

## 4. Bug 演练

三个 bug，各看一个症状。


In [ ]:
# BUG 1：符号写反 —— loss 也能降，但学的是反偏好
torch.manual_seed(0)
policy = TinyLM(); ref = copy.deepcopy(policy)
for p in ref.parameters(): p.requires_grad_(False)
opt = torch.optim.Adam(policy.parameters(), lr=5e-3)
for step in range(120):
    (ids_w, lab_w), (ids_l, lab_l) = make_batch()
    pi_w = get_batch_logps(policy(ids_w), lab_w); pi_l = get_batch_logps(policy(ids_l), lab_l)
    with torch.no_grad():
        ref_w = get_batch_logps(ref(ids_w), lab_w); ref_l = get_batch_logps(ref(ids_l), lab_l)
    loss, cr, rr = dpo_loss(pi_l, pi_w, ref_l, ref_w, beta=0.1)   # ← 入参顺序反了
    opt.zero_grad(); loss.backward(); opt.step()
# 用正确参数顺序在新 batch 上评估，才是真实的 reward accuracy
(ids_w, lab_w), (ids_l, lab_l) = make_batch()
with torch.no_grad():
    pi_w = get_batch_logps(policy(ids_w), lab_w); pi_l = get_batch_logps(policy(ids_l), lab_l)
    ref_w = get_batch_logps(ref(ids_w), lab_w); ref_l = get_batch_logps(ref(ids_l), lab_l)
    _, cr, rr = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1)
print('BUG 1 症状: 正确口径的 reward accuracy =', round((cr > rr).float().mean().item(), 2), '（< 0.5 = 在学反偏好）')


**修复 1**：把入参顺序改回来 —— `dpo_loss(pi_w, pi_l, ref_w, ref_l)`，chosen 在前。

**为什么有效**：DPO 的 loss 只认 margin 的方向，顺序反了优化方向就反了，但 loss 数值照样下降，所以这类 bug 只能靠 accuracy 抓。下面用正确顺序重训，accuracy 应该回到接近 1。

In [ ]:
# 修复版 1：正确入参顺序重训
torch.manual_seed(0)
policy = TinyLM(); ref = copy.deepcopy(policy)
for pp in ref.parameters(): pp.requires_grad_(False)
opt = torch.optim.Adam(policy.parameters(), lr=5e-3)
for step in range(120):
    (ids_w, lab_w), (ids_l, lab_l) = make_batch()
    pi_w = get_batch_logps(policy(ids_w), lab_w); pi_l = get_batch_logps(policy(ids_l), lab_l)
    with torch.no_grad():
        ref_w = get_batch_logps(ref(ids_w), lab_w); ref_l = get_batch_logps(ref(ids_l), lab_l)
    loss, cr, rr = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1)
    opt.zero_grad(); loss.backward(); opt.step()
print('FIX 1: reward accuracy =', round((cr > rr).float().mean().item(), 2), '（回到 > 0.5 = 偏好方向对了）')

In [ ]:
# BUG 2：mean 代替 sum —— 长序列被系统性打折（纯数值对比，不训练）
logits = torch.zeros(1, 6, 8)  # 均匀分布
long_lab = torch.tensor([[1, 2, 3, 4, 5, 0]])    # 5 个有效 token
short_lab = torch.tensor([[1, -100, -100, -100, -100, -100]])  # 1 个有效 token
s = get_batch_logps(logits, long_lab)
lg = logits[:, :-1]; lb = long_lab[:, 1:]
lp = torch.gather(F.log_softmax(lg, -1), 2, lb.clamp(min=0).unsqueeze(-1)).squeeze(-1)
m_long = lp.sum() / (lb != -100).sum()
print('BUG 2 症状: 长序列 sum =', round(s.item(), 2), ' mean =', round(m_long.item(), 2))
print('→ mean 让不同长度的序列不可比：训练时会系统性惩罚长 chosen，模型学成只说短句')


**修复 2**：log-prob 一律对 mask 后的 token **求和**，不做长度平均。

**为什么有效**：序列 log-prob 的定义就是各 token log-prob 之和（概率连乘取对数）。mean 相当于把每个序列的分数除以自己的长度，长回答被系统性压低。长度归一化是另一个算法决定（某些变体显式做），不能默认混进来。

In [ ]:
# 修复版 2：sum 口径下，长序列的分数不再被打折
def get_logps_mean(logits, labels):
    """反面教材版：入参、形状约定与 get_batch_logps 完全相同，
    但返回"每 token 平均"的 logp。只用于 BUG 2 的口径对照，真实训练不要用。"""
    logits = logits[:, :-1, :]; labels = labels[:, 1:]
    lp = torch.gather(F.log_softmax(logits, -1), 2, labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    mask = (labels != -100).float()
    return (lp * mask).sum(-1) / mask.sum(-1).clamp(min=1)

logits = torch.zeros(1, 6, 8)
long_lab = torch.tensor([[0, 1, 2, 3, 4, 5]])
s = get_batch_logps(logits, long_lab).item()
m = get_logps_mean(logits, long_lab).item()
print(f'FIX 2: 同一序列 sum = {s:.2f}（随长度增长，概率连乘的真实口径） vs mean = {m:.2f}（每 token 平均，与长度无关）')
print('→ 训练里混用 mean，等价于给长 chosen 暗中降权，模型会学成只说短句')

In [ ]:
# BUG 3：ref 没 detach（用 policy 自己当 ref）—— loss 冻在 ln2
# 入参 ref_* 全部换成 pi_* 本人：implicit reward 恒为 0
pi_w = torch.tensor([0.3]); pi_l = torch.tensor([-0.2])
loss, cr, rr = dpo_loss(pi_w, pi_l, pi_w, pi_l, beta=0.1)
print('BUG 3 症状: loss =', round(loss.item(), 4), '（= ln2 =', round(0.6931, 4), '，永远不动）')


**修复 3**：ref 必须是**独立的一份冻结模型**：单独 deepcopy、`requires_grad_(False)` 或全程 `no_grad`、绝不进 optimizer。

**为什么有效**：implicit reward 是 policy 相对 ref 的偏移，ref 跟着动，偏移恒为零、锚就没了。下面给一组真实的 policy/ref log-prob，loss 立刻离开 ln2、且对 policy 有梯度。

In [ ]:
# 修复版 3：独立冻结的 ref，loss 离开 ln2 且梯度正常
pi_w = torch.tensor([0.3], requires_grad=True); pi_l = torch.tensor([-0.2])
ref_w = torch.tensor([0.1]); ref_l = torch.tensor([0.0])   # 独立常数，不带梯度
loss, cr, rr = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1)
loss.backward()
print(f'FIX 3: loss = {loss.item():.4f}（≠ ln2），chosen reward = {cr.item():.3f}，policy 梯度 = {pi_w.grad.item():.4f}（非零）')

## 5. Debug 速查（现场排查顺序）

accuracy → margin 符号 → loss 是否冻在 0.693 → 检查 mask 里 prompt 有没有漏进来 → 检查 logits/labels 有没有 shift 对齐。

三类 bug 的修复都写在各自正下方了（BUG 1 入参顺序、BUG 2 sum 不是 mean、BUG 3 ref 独立冻结），这里只留顺序，不重复内容。

## 小结

- DPO 的命门不在公式，在 log-prob：shift 对齐、mask 干净、**sum 不是 mean**、ref 独立冻结。
- 三个 bug 三个指纹：acc<0.5 / 偏好短句 / loss=ln2。
- DPO 不碰采样，所以它的天花板是数据分布 —— chosen/rejected 之外的行为不归它管（与 PPO/GRPO 的本质区别，下一篇展开）。

**下一篇**：Research 03 — PPO/GRPO loss、clip、advantage、KL
